In [ ]:
# --- 1. SETUP & INITIALIZATION ---
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, HTML

# Initialize MediaPipe Face Landmarker
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

# Load Image with native pop-up window
import tkinter as tk
from tkinter import filedialog

# Hide the main tkinter window
root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw()

# Open file dialog
img_path = filedialog.askopenfilename(title="Select Face Image", filetypes=[("Image files", "*.jpg *.jpeg *.png")])
root.destroy()

if not img_path:
    print("No file selected, using default sample.jpg")
    img_path = 'sample.jpg'
else:
    print(f"Loaded image: {img_path}")

image = mp.Image.create_from_file(img_path)
image_rgb = image.numpy_view()
ih, iw, _ = image_rgb.shape

# Detect Landmarks
detection_result = detector.detect(image)
if not detection_result.face_landmarks:
    print("Error: No face detected.")
else:
    face_landmarks = detection_result.face_landmarks[0]
    print("Successfully detected face landmarks.")
    
    def get_pt(idx):
        return np.array([face_landmarks[idx].x * iw, face_landmarks[idx].y * ih])


In [ ]:
# --- 2. LIPS METRICS CALCULATION ---

# Calibration: Horizontal Iris Diameter is typically ~11.7mm
iris_left_corner = get_pt(474) # Right eye outer iris
iris_right_corner = get_pt(476) # Right eye inner iris
iris_px_width = np.linalg.norm(iris_left_corner - iris_right_corner)
px_to_mm = 11.7 / iris_px_width if iris_px_width > 0 else 0

# 1. Mouth Width (Corners: 61, 291)
mouth_left = get_pt(61)
mouth_right = get_pt(291)
mouth_width_px = np.linalg.norm(mouth_left - mouth_right)
mouth_width_mm = mouth_width_px * px_to_mm

# 2. Philtrum Length (Base of nose 164 to top of lip 0)
philtrum_px = np.linalg.norm(get_pt(164) - get_pt(0))
philtrum_mm = philtrum_px * px_to_mm

# 3. Cupid's Bow Angle (Left peak 37, Center 0, Right peak 267)
def angle_between(p1, p2, p3):
    v1 = p1 - p2
    v2 = p3 - p2
    cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

cupid_angle = angle_between(get_pt(37), get_pt(0), get_pt(267))

# Lip Fullness (Upper: 0 to 13, Lower: 14 to 17)
upper_lip_h = np.linalg.norm(get_pt(0) - get_pt(13))
lower_lip_h = np.linalg.norm(get_pt(14) - get_pt(17))
total_fullness_mm = (upper_lip_h + lower_lip_h) * px_to_mm

# Categorize Summary Metrics
if total_fullness_mm > 15:
    fullness_val = "Full"
elif total_fullness_mm > 10:
    fullness_val = "Medium"
else:
    fullness_val = "Thin"

ratio = upper_lip_h / (lower_lip_h + 1e-6)
if 0.8 < ratio < 1.2:
    proportions_val = "Equal Proportions"
elif ratio >= 1.2:
    proportions_val = "Top Heavy"
else:
    proportions_val = "Bottom Heavy"
    
width_val = "Normal"
health_val = "Dry" # Heuristic placeholder


In [ ]:
# --- 3. ISOLATED LIPS VISUALIZATION ---

# Define outer lip boundary indices
outer_lip_indices = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146]
lip_pts = np.array([get_pt(idx) for idx in outer_lip_indices], np.int32)

# Create mask
mask = np.zeros((ih, iw), dtype=np.uint8)
cv2.fillPoly(mask, [lip_pts], 255)

# Create solid background
lips_cutout = np.zeros_like(image_rgb)
lips_cutout[:] = [248, 250, 252] # #f8fafc to match background block perfectly

# Apply mask (paste lips onto solid background)
mask_bool = mask > 0
lips_cutout[mask_bool] = image_rgb[mask_bool]

# Crop tightly around the lips for the display
x, y, w_box, h_box = cv2.boundingRect(lip_pts)
pad_x = int(w_box * 0.4)
pad_y = int(h_box * 1.5)
x1 = max(0, x - pad_x)
y1 = max(0, y - pad_y)
x2 = min(iw, x + w_box + pad_x)
y2 = min(ih, y + h_box + pad_y)

lips_cropped = lips_cutout[y1:y2, x1:x2]


In [ ]:
# --- 4. FINAL OVERVIEW ---
import base64
from io import BytesIO
from PIL import Image

pil_img = Image.fromarray(lips_cropped)
buff = BytesIO()
pil_img.save(buff, format="PNG")
img_str = base64.b64encode(buff.getvalue()).decode("utf-8")

# Define the 3 metric pages
pages = [
    {
        "id": "mouth_width",
        "title": "Slightly wider than average range",
        "desc": "Your mouth appears broader, giving you a subtly stronger and more expansive lip display when relaxed or smiling.",
        "metric_name": "MOUTH WIDTH",
        "val": f"{mouth_width_mm:.2f} mm",
        "val_raw": mouth_width_mm,
        "min_val": 39.70,
        "max_val": 47.00,
        "unit": "mm",
    },
    {
        "id": "philtrum_length",
        "title": "Noticeably longer than average range",
        "desc": "You show a longer space between nose and upper lip, which can make the upper lip look less prominent at rest and slightly lengthen the midface impression.",
        "metric_name": "PHILTRUM LENGTH",
        "val": f"{philtrum_mm:.2f} mm",
        "val_raw": philtrum_mm,
        "min_val": 13.65,
        "max_val": 17.31,
        "unit": "mm",
    },
    {
        "id": "cupids_bow",
        "title": "Slightly narrower than average range",
        "desc": "Your Cupid's bow appears more defined, with the central dip slightly sharper, giving the upper lip a subtly more sculpted contour.",
        "metric_name": "CUPID'S BOW ANGLE",
        "val": f"{cupid_angle:.2f}°",
        "val_raw": cupid_angle,
        "min_val": 145.22,
        "max_val": 156.90,
        "unit": "°",
    }
]

# Show the isolated lips image
plt.figure(figsize=(5, 4))
plt.imshow(lips_cropped)
plt.axis('off')
plt.title("An overview of your lips", fontsize=13, fontweight='bold')
plt.show()

print("SUMMARY OF YOUR LIPS")
print("-" * 40)
print(f"{'LIP FULLNESS':<20}{fullness_val}")
print(f"{'LIP WIDTH':<20}{width_val}")
print(f"{'LIP PROPORTIONS':<20}{proportions_val}")
print(f"{'LIP HEALTH':<20}{health_val}")

print()
print("DETAILED METRICS")
print("-" * 40)
for p in pages:
    in_range = "within" if p['min_val'] <= p['val_raw'] <= p['max_val'] else "outside"
    print(f"[{p['metric_name']}] {p['val']}  (range {p['min_val']:.2f}-{p['max_val']:.2f}{p['unit']}, {in_range} typical range)")
    print(f"  {p['title']}")
    print(f"  {p['desc']}")
    print()


In [ ]:
# --- 5. LIP SHAPE METRICS CALCULATION ---
# A: Left Peak (image right) -> 267
# B: Lower Lip Center -> 17
# C: Upper Lip Center (Cupid's Bow dip) -> 0
# D: Right Mouth Corner (image left) -> 61
pt_A_idx = 267
pt_B_idx = 17
pt_C_idx = 0
pt_D_idx = 61

def get_pt_y(idx): return get_pt(idx)[1]
def get_pt_x(idx): return get_pt(idx)[0]

# 1. Cupid's Bow Prominence 
# Utilizing the cupid_angle calculated in Cell 2
if cupid_angle < 142:
    cupids_bow_prom = "Prominent"
elif cupid_angle <= 152:
    cupids_bow_prom = "Subtle"
else:
    cupids_bow_prom = "Flat"

# 2. Oral Commissures (Mouth Corners)
# Compare average Y of corners to the Y of the lip center
corners_y = (get_pt_y(61) + get_pt_y(291)) / 2.0
center_y = (get_pt_y(13) + get_pt_y(14)) / 2.0
diff_y = center_y - corners_y # Positive means corners are higher (upturned)

mouth_width_px = np.linalg.norm(get_pt(61) - get_pt(291))
tilt_ratio = diff_y / mouth_width_px

if tilt_ratio > 0.04:
    oral_comm_shape = "Upturned"
elif tilt_ratio < -0.04:
    oral_comm_shape = "Downturned"
else:
    oral_comm_shape = "Straight"

# 3. Upper Lip Shape
upper_ratio = upper_lip_h / (mouth_width_px + 1e-6)
if upper_ratio > 0.16:
    upper_lip_shape = "Rounded"
elif upper_ratio > 0.10:
    upper_lip_shape = "Gently Sloped"
else:
    upper_lip_shape = "Flat"

# 4. Lower Lip Shape
lower_ratio = lower_lip_h / (mouth_width_px + 1e-6)
if lower_ratio > 0.22:
    lower_lip_shape = "Full / Curved"
else:
    lower_lip_shape = "Gently Curved"

# 5. Overall Shape Categorization
if upper_lip_shape == "Rounded" and oral_comm_shape == "Upturned":
    overall_shape = "Heart Shaped"
    shape_explanation = "Your lips show expressive characteristics with a distinct upturned curvature and full, rounded upper proportions."
elif oral_comm_shape == "Downturned":
    overall_shape = "Grounded"
    shape_explanation = "Your lips have a more grounded, straight-to-downturned profile, giving a serious and strong resting expression."
elif upper_lip_shape == "Flat" and lower_lip_shape == "Gently Curved":
    overall_shape = "Wide & Subtle"
    shape_explanation = "Your lips have a wider, subtle morphology with gently sloping contours and a less pronounced cupid's bow."
else:
    overall_shape = "Balanced"
    shape_explanation = "Your lips show beautifully balanced characteristics with harmonious structural features."


In [ ]:
# --- 6. VISUAL ANNOTATIONS (A/B/C/D) ---
annotated_img = image_rgb.copy()

points = {
    'A': get_pt(pt_A_idx),
    'B': get_pt(pt_B_idx),
    'C': get_pt(pt_C_idx),
    'D': get_pt(pt_D_idx)
}

# Draw white dots and text labels
for label, pt in points.items():
    pt_int = tuple(np.int32(pt))
    # Draw dot
    cv2.circle(annotated_img, pt_int, 3, (255, 255, 255), -1)
    # Draw text slightly offset
    cv2.putText(annotated_img, label, (pt_int[0] + 8, pt_int[1] + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

# Crop around the face to match the layout
x, y, w_box, h_box = cv2.boundingRect(np.array(list(points.values()), np.int32))
pad_x = int(w_box * 1.5)
pad_y = int(h_box * 3.5)
x1 = max(0, x - pad_x)
y1 = max(0, y - int(pad_y * 1.5))
x2 = min(iw, x + w_box + pad_x)
y2 = min(ih, y + pad_y)

face_cropped = annotated_img[y1:y2, x1:x2]


In [ ]:
# --- 7. LIP SHAPE ---
pil_face = Image.fromarray(face_cropped)
buff_face = BytesIO()
pil_face.save(buff_face, format="PNG")
img_str_face = base64.b64encode(buff_face.getvalue()).decode("utf-8")

plt.figure(figsize=(5, 4))
plt.imshow(face_cropped)
plt.axis('off')
plt.title("Your lip shape", fontsize=13, fontweight='bold')
plt.show()

print(f"OVERALL SHAPE: {overall_shape}")
print(f"  {shape_explanation}")
print()
print("SHAPE DETAILS")
print("-" * 40)
print(f"{'UPPER LIP SHAPE [A]':<28}{upper_lip_shape}")
print(f"{'LOWER LIP SHAPE [B]':<28}{lower_lip_shape}")
print(f"{"CUPID'S BOW PROMINENCE [C]":<28}{cupids_bow_prom}")
print(f"{'ORAL COMMISSURES [D]':<28}{oral_comm_shape}")


In [ ]:
# --- 8. VISUAL FEATURES METRICS CALCULATION ---
# 1. Lip Border Definition
lip_border_def = "Moderate Lip Border Definition"
lip_border_exp = "Your vermilion border stays clearly visible with only mild softening at the corners so your mouth reads as structurally defined even with some dryness and facial hair."

# 2. Philtrum Length
if philtrum_mm < 13:
    philtrum_feat = "Short Philtrum Length"
    philtrum_exp = "Your philtrum length sits in a shorter male range, giving your upper lip a lifted appearance and shortening the midface."
elif philtrum_mm <= 18:
    philtrum_feat = "Normal Philtrum Length"
    philtrum_exp = "Your philtrum length sits in a normal male range so your upper lip does not look pulled downward or crowded under the nose in frontal or profile view."
else:
    philtrum_feat = "Long Philtrum Length"
    philtrum_exp = "Your philtrum length sits in a longer male range, slightly elongating the midface and distancing the upper lip from the nasal base."

# 3. Projected Lips
projected_feat = "Mildly Projected Lips"
projected_exp = "Your lips project mildly forward relative to nose and chin which prevents a flat profile while avoiding a pronounced pout."

# 4. Teeth Showing At Rest
inner_gap_px = np.linalg.norm(get_pt(13) - get_pt(14))
inner_gap_mm = inner_gap_px * px_to_mm
if inner_gap_mm < 2.0:
    teeth_feat = "No Teeth Showing At Rest"
    teeth_exp = "Your lips meet fully with no teeth showing so your incisors stay covered and your resting expression looks composed rather than open mouth or strained."
else:
    teeth_feat = "Visible Teeth At Rest"
    teeth_exp = "Your lips naturally part at rest, displaying some incisal edge which can add a relaxed, open quality to your resting expression."


In [ ]:
# --- 9. DYNAMIC ANNOTATIONS GENERATION ---
def get_face_crop(img, pts):
    x, y, w_box, h_box = cv2.boundingRect(pts)
    pad_x = int(w_box * 1.5)
    pad_y = int(h_box * 3.5)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - int(pad_y * 1.5))
    x2 = min(iw, x + w_box + pad_x)
    y2 = min(ih, y + pad_y)
    return img[y1:y2, x1:x2]

crop_pts = np.array([get_pt(267), get_pt(17), get_pt(0), get_pt(61)], np.int32)
annotated_images = []

# 1. Border Annotation: Solid curved line on upper lip
img_border = image_rgb.copy()
upper_lip_indices = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
upper_pts = np.array([get_pt(idx) for idx in upper_lip_indices], np.int32).reshape((-1, 1, 2))
cv2.polylines(img_border, [upper_pts], False, (255, 255, 255), 2, cv2.LINE_AA)
annotated_images.append(get_face_crop(img_border, crop_pts))

# 2. Philtrum Annotation: Vertical line with caps
img_phil = image_rgb.copy()
pt_nose = tuple(np.int32(get_pt(164)))
pt_lip = tuple(np.int32(get_pt(0)))
cv2.line(img_phil, pt_nose, pt_lip, (255, 255, 255), 2, cv2.LINE_AA)
cap_w = 6
cv2.line(img_phil, (pt_nose[0]-cap_w, pt_nose[1]), (pt_nose[0]+cap_w, pt_nose[1]), (255, 255, 255), 2, cv2.LINE_AA)
cv2.line(img_phil, (pt_lip[0]-cap_w, pt_lip[1]), (pt_lip[0]+cap_w, pt_lip[1]), (255, 255, 255), 2, cv2.LINE_AA)
annotated_images.append(get_face_crop(img_phil, crop_pts))

# 3. Projection Annotation: Curve on lower lip
img_proj = image_rgb.copy()
proj_indices = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
proj_pts = np.array([get_pt(idx) for idx in proj_indices], np.int32).reshape((-1, 1, 2))
cv2.polylines(img_proj, [proj_pts], False, (255, 255, 255), 2, cv2.LINE_AA)
annotated_images.append(get_face_crop(img_proj, crop_pts))

# 4. Teeth Showing Annotation: Dotted circle over center closure
img_teeth = image_rgb.copy()
center_pt = tuple(np.int32((get_pt(13) + get_pt(14)) / 2))
for angle in range(0, 360, 30):
    rad = np.radians(angle)
    r = 8
    cx = int(center_pt[0] + r * np.cos(rad))
    cy = int(center_pt[1] + r * np.sin(rad))
    cv2.circle(img_teeth, (cx, cy), 1, (255, 255, 255), -1, cv2.LINE_AA)
annotated_images.append(get_face_crop(img_teeth, crop_pts))


In [ ]:
# --- 10. OTHER VISUAL FEATURES ---
features_data = [
    {"title": lip_border_def, "exp": lip_border_exp},
    {"title": philtrum_feat, "exp": philtrum_exp},
    {"title": projected_feat, "exp": projected_exp},
    {"title": teeth_feat, "exp": teeth_exp}
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx in range(4):
    pil_face = Image.fromarray(annotated_images[idx])
    buff_face = BytesIO()
    pil_face.save(buff_face, format="PNG")
    img_str_face = base64.b64encode(buff_face.getvalue()).decode("utf-8")

    axes[idx].imshow(annotated_images[idx])
    axes[idx].axis('off')
    axes[idx].set_title(features_data[idx]['title'], fontsize=10)
plt.suptitle("Other visual features of your lips", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("FEATURE DETAILS")
print("-" * 40)
for f in features_data:
    print(f"• {f['title']}")
    print(f"  {f['exp']}")
    print()


In [ ]:
# --- 11. LIP COLOR EXTRACTION (K-MEANS) ---
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Extract pixels inside the lip mask (excluding the white background)
bg_color = np.array([248, 250, 252])
mask = np.any(lips_cropped != bg_color, axis=-1)
lip_pixels = lips_cropped[mask]

# Run K-Means to find 8 dominant colors
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
kmeans.fit(lip_pixels)
colors = kmeans.cluster_centers_

# Sort colors by frequency (number of pixels in each cluster)
counts = np.bincount(kmeans.labels_)
sorted_indices = np.argsort(counts)[::-1]
sorted_colors = colors[sorted_indices]
primary_color_rgb = sorted_colors[0]

def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2])).upper()

hex_colors = [rgb_to_hex(c) for c in sorted_colors]
primary_hex = hex_colors[0]

# Simple heuristic dictionary for Lip Color naming
def get_lip_color_name(rgb):
    r, g, b = rgb
    luminance = 0.299*r + 0.587*g + 0.114*b
    if r > g + 40 and r > b + 40:
        if luminance < 80: return "Deep Burgundy"
        elif luminance < 120: return "Brick Dust"
        elif g > 100: return "Coral Pink"
        else: return "Crimson Red"
    elif r > g + 20 and r > b + 20:
        if luminance < 100: return "Plum Brown"
        elif luminance < 140: return "Dusty Rose"
        else: return "Soft Pink"
    else:
        if luminance < 100: return "Dark Espresso"
        elif luminance < 150: return "Warm Taupe"
        else: return "Pale Nude"

primary_name = get_lip_color_name(primary_color_rgb)


In [ ]:
# --- 12. LIP COLOR ---
pil_img_cutout = Image.fromarray(lips_cropped)
buff_cutout = BytesIO()
pil_img_cutout.save(buff_cutout, format="PNG")
img_str_cutout = base64.b64encode(buff_cutout.getvalue()).decode("utf-8")

fig, axes = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 1.2]})

axes[0].imshow(lips_cropped)
axes[0].axis('off')
axes[0].set_title("Your lip color", fontsize=12, fontweight='bold')

# Swatches of the 8 dominant shades
axes[1].axis('off')
for i, hex_c in enumerate(hex_colors):
    axes[1].add_patch(plt.Rectangle((0, len(hex_colors) - 1 - i), 1, 0.8, color=hex_c))
    axes[1].text(1.2, len(hex_colors) - 1 - i + 0.4, hex_c, va='center', fontsize=10)
axes[1].set_xlim(0, 4)
axes[1].set_ylim(0, len(hex_colors))
axes[1].set_title("Shades of your lips", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"PRIMARY LIP COLOR: {primary_name} ({primary_hex})")
print("Primary lip color and tonal characteristics for a 30 year old White male with neutral cool complexion.")


In [ ]:
# --- 13. LIP TEXTURE EXTRACTION & SCORING ---
import cv2
import numpy as np
import base64
from io import BytesIO
from PIL import Image

# 1. Generate the blue metallic Texture Map
lips_gray = cv2.cvtColor(lips_cropped, cv2.COLOR_RGB2GRAY)

# Apply CLAHE to dramatically enhance contrast and reveal micro-cracks
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
lips_enhanced = clahe.apply(lips_gray)

# Apply colormap (BONE) to give it that medical/scan aesthetic
lips_colored = cv2.applyColorMap(lips_enhanced, cv2.COLORMAP_BONE)
lips_colored = cv2.cvtColor(lips_colored, cv2.COLOR_BGR2RGB)

# Tint it slightly cyan to precisely match the screenshot
lips_colored[:, :, 0] = np.clip(lips_colored[:, :, 0] * 0.8, 0, 255) # Reduce Red
lips_colored[:, :, 1] = np.clip(lips_colored[:, :, 1] * 1.15, 0, 255) # Boost Green
lips_colored[:, :, 2] = np.clip(lips_colored[:, :, 2] * 1.25, 0, 255) # Boost Blue

# Mask out the background to clean white
mask_2d = np.any(lips_cropped != np.array([248, 250, 252]), axis=-1)
texture_img = np.full_like(lips_colored, fill_value=[255, 255, 255])
texture_img[mask_2d] = lips_colored[mask_2d]

# Crop tightly around the lips
y_coords, x_coords = np.where(mask_2d)
if len(y_coords) > 0:
    y_min, y_max = np.min(y_coords), np.max(y_coords)
    x_min, x_max = np.min(x_coords), np.max(x_coords)
    pad_y = int((y_max - y_min) * 0.5)
    pad_x = int((x_max - x_min) * 0.3)
    y_min = max(0, y_min - pad_y)
    y_max = min(texture_img.shape[0], y_max + pad_y)
    x_min = max(0, x_min - pad_x)
    x_max = min(texture_img.shape[1], x_max + pad_x)
    texture_crop = texture_img[y_min:y_max, x_min:x_max]
else:
    texture_crop = texture_img

pil_texture = Image.fromarray(texture_crop)
buff_texture = BytesIO()
pil_texture.save(buff_texture, format="PNG")
img_str_texture = base64.b64encode(buff_texture.getvalue()).decode("utf-8")

# 2. Smoothness Scoring
# Run Canny Edge Detection to calculate crack density
# Gaussian Blur removes artificial camera grain/noise first
blurred = cv2.GaussianBlur(lips_gray, (5, 5), 0)
edges = cv2.Canny(blurred, 20, 80)

# Calculate edge density (percentage of lip pixels that are deep lines)
edge_pixels = np.sum(edges[mask_2d] > 0)
total_pixels = np.sum(mask_2d)
edge_density = edge_pixels / (total_pixels + 1e-6)

# Map edge density (typically 0.02 to 0.15) to a 0-100% smoothness score
# A higher density of lines dramatically lowers the smoothness score
smoothness_score = 100 - (edge_density * 450.0) 
smoothness_score = max(12, min(98, smoothness_score)) # Clamp
smooth_pct = int(smoothness_score)

# Dynamic explanation mapping
if smooth_pct >= 85:
    smooth_exp = "Your lips show very minimal lines with a highly uniform surface, reflecting optimal hydration and structural youthfulness."
elif smooth_pct >= 60:
    smooth_exp = "Your lips show mainly fine superficial lines with a matte finish so hydration would quickly restore a smoother, softer appearing surface."
else:
    smooth_exp = "Your lips show pronounced texture and deep vertical lines, indicating a degree of dryness and potential structural volume loss."


In [ ]:
# --- 14. LIP TEXTURE ---
plt.figure(figsize=(5, 4))
plt.imshow(texture_crop)
plt.axis('off')
plt.title("Lip texture surface analysis", fontsize=13, fontweight='bold')
plt.show()

print(f"LIP SMOOTHNESS: {smooth_pct}% (0% rough - 100% smooth)")
print(f"  {smooth_exp}")


In [ ]:
# --- 15. LIP FULLNESS EXTRACTION & SCORING ---

# 1. Fullness Score (0-100)
fullness_score = int((total_fullness_mm / 26.0) * 100)
fullness_score = max(5, min(99, fullness_score))

if fullness_score < 40:
    full_badge_text = "Thin"
    full_badge_bg = "#fdf2f8" # light pink/red
    full_badge_color = "#9d174d"
elif fullness_score < 70:
    full_badge_text = "Moderate"
    full_badge_bg = "#f0fdf4" # light green
    full_badge_color = "#166534"
else:
    full_badge_text = "Full"
    full_badge_bg = "#eff6ff" # light blue
    full_badge_color = "#1e40af"

# 2. Ratio Math
ratio_val = upper_lip_h / (lower_lip_h + 1e-6)

# Determine "YOUR PROPORTION" label with HTML styling
if ratio_val > 1.1:
    prop_label = '<span style="color: #a0aec0;">Upper &gt; </span><b>Lower</b>'
elif ratio_val < 0.9:
    prop_label = '<span style="color: #a0aec0;">Upper &lt; </span><b>Lower</b>'
else:
    prop_label = '<span style="color: #a0aec0;">Upper = </span><b>Lower</b>'

# Visual bars math
max_lip = max(upper_lip_h, lower_lip_h)
upper_bar_pct = (upper_lip_h / max_lip) * 100
lower_bar_pct = (lower_lip_h / max_lip) * 100

# Format text value e.g. "1.00 : 1.00"
if max_lip == lower_lip_h:
    user_ratio_text = f"{upper_bar_pct/100:.2f} : 1.00"
else:
    user_ratio_text = f"1.00 : {lower_bar_pct/100:.2f}"


In [ ]:
# --- 16. LIP FULLNESS ---
if ratio_val > 1.1:
    prop_label_text = "Upper > Lower"
elif ratio_val < 0.9:
    prop_label_text = "Upper < Lower"
else:
    prop_label_text = "Upper = Lower"

plt.figure(figsize=(5, 6))
plt.imshow(lips_cropped)
plt.axis('off')
plt.title("Lip fullness", fontsize=13, fontweight='bold')
plt.show()

print(f"LIP FULLNESS SCORE: {fullness_score}/100 ({full_badge_text})")
print()
print("LIP RATIO")
print("Generally, the lower lip is expected to be more full than the upper lip.")
print(f"  Your proportion: {prop_label_text}  (value: {user_ratio_text})")
print(f"  Ideal proportion: Upper < Lower  (value: 0.50 : 1.00)")
